In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

# Загрузка данных

In [2]:
train_data = pd.read_csv("../input_files/train.csv")
test_data = pd.read_csv("../input_files/test.csv")

# Бейзлайн

In [3]:
train_data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


берем признаки Pclass, Sex, Age, SibSp, Parch, Fare, Embarked

In [4]:
num_features = ['Age', 'SibSp', 'Parch', 'Fare']
cat_features = ['Sex', 'Pclass', 'Embarked']

features = num_features + cat_features

X_train = train_data[features]
y_train = train_data['Survived']

In [5]:
num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ohe', OneHotEncoder(handle_unknown='ignore'))
])

transformer = ColumnTransformer([
    ('num_pipeline', num_pipe, num_features),
    ('cat_pipeline', cat_pipe, cat_features),
])

model = Pipeline([
    ('preprocessing', transformer),
    ('lr', LogisticRegression())
])

In [6]:
cv = cross_val_score(
    estimator=model,
    X=X_train,
    y=y_train,
    cv=5,
    scoring='accuracy',
    error_score='raise'
)

In [7]:
print(f'CV scores: {cv}')
print(f'Mean accuracy: {cv.mean():.4f}')
print(f'Std accuracy: {cv.std():.4f}')

CV scores: [0.77653631 0.78651685 0.78651685 0.76966292 0.83146067]
Mean accuracy: 0.7901
Std accuracy: 0.0216


Сравним с классификатором, который предсказывает самый частый класс

In [8]:
from sklearn.dummy import DummyClassifier

dummy_clf = DummyClassifier(strategy="most_frequent")

dummy_cv = cross_val_score(
    DummyClassifier(strategy='most_frequent'),
    X_train,
    y_train,
    cv=5,
    scoring='accuracy'
)

print(f'Dummy accuracy: {dummy_cv.mean():.4f}')

Dummy accuracy: 0.6162


# Эксперименты с применением feature engineering

In [9]:
def evaluate_features(X, num_features, cat_features, y, cv=5):
    transformer = ColumnTransformer([
        ('num', num_pipe, num_features),
        ('cat', cat_pipe, cat_features)
    ])

    model = Pipeline([
        ('preprocessing', transformer),
        ('lr', LogisticRegression())
    ])

    scores = cross_val_score(
        model, X, y,
        cv=cv,
        scoring='accuracy'
    )

    return scores.mean(), scores.std()

### E1: исходные фичи + IsChild

In [10]:
X_exp = X_train.copy()

X_exp['IsChild'] = np.where(
    X_exp['Age'].isna(),
    np.nan,
    (X_exp['Age'] <= 12).astype(int)
)

num_features = ['Age', 'SibSp', 'Parch', 'Fare', 'IsChild']
cat_features = ['Sex', 'Pclass', 'Embarked']

print(evaluate_features(X_exp, num_features, cat_features, y_train))

(np.float64(0.8092021844203126), np.float64(0.02383888168804701))


### E2: объединяем SibSp и Parch в FamilySize

In [11]:
X_exp = X_train.copy()

X_exp['FamilySize'] = X_exp['SibSp'] + X_exp['Parch'] + 1

num_features = ['Age', 'Fare', 'FamilySize']
cat_features = ['Sex', 'Pclass', 'Embarked']

print(evaluate_features(X_exp, num_features, cat_features, y_train))

(np.float64(0.7957629778419435), np.float64(0.021777864878186445))


### E3: исходные + IsAlone

In [12]:
X_exp = X_train.copy()

X_exp['IsAlone'] = ((X_exp['SibSp'] + X_exp['Parch']) == 0).astype(int)

num_features = ['Age', 'Fare', 'Parch', 'SibSp', 'IsAlone']
cat_features = ['Sex', 'Pclass', 'Embarked']

print(evaluate_features(X_exp, num_features, cat_features, y_train))

(np.float64(0.7946393823363254), np.float64(0.01867102164858746))


### E4: + размер семьи (малый, средний, большой)

In [13]:
X_exp = X_train.copy()

X_exp['FamilySize'] = X_exp['SibSp'] + X_exp['Parch'] + 1

X_exp['FamilyGroup'] = pd.cut(
    X_exp['FamilySize'],
    bins=[0, 1, 4, float('inf')],
    labels=['Alone', 'Small', 'Large']
)

X_exp = X_exp.drop(columns=['SibSp', 'Parch', 'FamilySize'])

num_features = ['Age', 'Fare']
cat_features = ['Sex', 'Pclass', 'Embarked', 'FamilyGroup']

print(evaluate_features(X_exp, num_features, cat_features, y_train))

(np.float64(0.8069487163392128), np.float64(0.009968828857687122))


### E5: извлекаем титул из имени

In [14]:
X_exp = train_data.copy()

X_exp['Title'] = (
    X_exp['Name']
    .str.extract(r',\s*([^.]*)\.')[0]
    .str.strip()
)

X_exp['Title'].value_counts()

Title
Mr              517
Miss            182
Mrs             125
Master           40
Dr                7
Rev               6
Major             2
Mlle              2
Col               2
Don               1
Mme               1
Ms                1
Lady              1
Sir               1
Capt              1
the Countess      1
Jonkheer          1
Name: count, dtype: int64

In [15]:
X_exp['Title'] = X_exp['Title'].replace({
    'Mme': 'Mrs',
    'Mile': 'Miss',
    'Ms': 'Miss',
})

common_titles = ['Mr', 'Miss', 'Mrs', 'Master']

X_exp['Title'] = X_exp['Title'].where(
    X_exp['Title'].isin(common_titles),
    'Rare',
)

In [16]:
num_features = ['Age', 'Fare', 'SibSp', 'Parch']
cat_features = ['Sex', 'Pclass', 'Title', 'Embarked']

print(evaluate_features(X_exp, num_features, cat_features, y_train))

(np.float64(0.8237775406440274), np.float64(0.022750887937693885))


### E6: Title + IsChild

In [17]:
X_exp = train_data.copy()

X_exp['Title'] = (
    X_exp['Name']
    .str.extract(r',\s*([^.]*)\.')[0]
    .str.strip()
)

X_exp['Title'] = X_exp['Title'].replace({
    'Mme': 'Mrs',
    'Mlle': 'Miss',
    'Ms': 'Miss',
})

common_titles = ['Mr', 'Miss', 'Mrs', 'Master']

X_exp['Title'] = X_exp['Title'].where(
    X_exp['Title'].isin(common_titles),
    'Rare',
)

X_exp['IsChild'] = np.where(
    X_exp['Age'].isna(),
    np.nan,
    (X_exp['Age'] <= 12).astype(int)
)

num_features = ['Age', 'Fare', 'SibSp', 'Parch', 'IsChild']
cat_features = ['Sex', 'Pclass', 'Title', 'Embarked']

print(evaluate_features(X_exp, num_features, cat_features, y_train))

(np.float64(0.8237900947837549), np.float64(0.025789214534588253))


одновременное использование Title и IsChild не улучшает метрику. Это я бы объяснил тем, что Title уже содержит в себе информацию о том, ребенок это или нет (класс Master -- обращение к мальчику)

### E7: Title + FamilyGroup

In [18]:
X_exp = train_data.copy()

X_exp['Title'] = (
    X_exp['Name']
    .str.extract(r',\s*([^.]*)\.')[0]
    .str.strip()
)

X_exp['Title'] = X_exp['Title'].replace({
    'Mme': 'Mrs',
    'Mlle': 'Miss',
    'Ms': 'Miss',
})

common_titles = ['Mr', 'Miss', 'Mrs', 'Master']

X_exp['Title'] = X_exp['Title'].where(
    X_exp['Title'].isin(common_titles),
    'Rare',
)

X_exp['FamilySize'] = X_exp['SibSp'] + X_exp['Parch'] + 1

X_exp['FamilyGroup'] = pd.cut(
    X_exp['FamilySize'],
    bins=[0, 1, 4, float('inf')],
    labels=['Alone', 'Small', 'Large']
)

X_exp = X_exp.drop(columns=['SibSp', 'Parch', 'FamilySize'])

num_features = ['Age', 'Fare']
cat_features = ['Sex', 'Pclass', 'Title', 'Embarked', 'FamilyGroup']

print(evaluate_features(X_exp, num_features, cat_features, y_train))

(np.float64(0.8237838177138912), np.float64(0.020974212551877137))


## E8: добавим признак HasCabin

In [19]:
X_exp = train_data.copy()

X_exp['Title'] = (
    X_exp['Name']
    .str.extract(r',\s*([^.]*)\.')[0]
    .str.strip()
)

X_exp['Title'] = X_exp['Title'].replace({
    'Mme': 'Mrs',
    'Mlle': 'Miss',
    'Ms': 'Miss',
})

common_titles = ['Mr', 'Miss', 'Mrs', 'Master']

X_exp['Title'] = X_exp['Title'].where(
    X_exp['Title'].isin(common_titles),
    'Rare',
)

X_exp['FamilySize'] = X_exp['SibSp'] + X_exp['Parch'] + 1

X_exp['FamilyGroup'] = pd.cut(
    X_exp['FamilySize'],
    bins=[0, 1, 4, float('inf')],
    labels=['Alone', 'Small', 'Large']
)

X_exp = X_exp.drop(columns=['FamilySize'])

X_exp['HasCabin'] = X_exp['Cabin'].notna().astype(int)

num_features = ['Age', 'Fare', 'SibSp', 'Parch']
cat_features = ['Sex', 'Pclass', 'Title', 'Embarked', 'FamilyGroup', 'HasCabin']

print(evaluate_features(X_exp, num_features, cat_features, y_train))

(np.float64(0.8305128366078713), np.float64(0.024033894746380253))


лучший результат